# 01. Data preparation

Download the pinned DataSEC release, index the clips, check the fixed split and cache the resampled waveforms used for training. Run once per machine; the cache is needed by `03_train_colab.ipynb`.

The raw audio is CC-BY-NC-SA and is **not** committed (`data/` is git-ignored).

*Generated by `scripts/build_notebooks.py` from the `.py` files; edit those and rebuild, not this notebook.*

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/DataSEC-Sound-Classification")  # <- where the repository is on Drive
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
# os.environ["ATI_CONFIG"] = "configs/default.yaml"  # optional: another config (read when src.utils is imported)

import pandas as pd
from IPython.display import Image, display

print("repository:", ROOT)

repository: C:\ATI\DataSEC-Sound-Classification-main


In [2]:
import subprocess

if IN_COLAB:  # torch, torchaudio, torchvision are preinstalled on Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soxr", "thop"], check=True)

## 1. Download and verify (6.4 GB)

Source: `scripts/download_data.py`

Download the pinned DataSEC release (Zenodo DOI 10.5281/zenodo.17033970), verify MD5, and extract.

Downloads in parallel byte-range chunks (resumable: finished chunks are skipped on re-run).

In [3]:
import argparse
import hashlib
import json
import time
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

In [4]:
RECORD_ID = "17033970"
DOI = "10.5281/zenodo.17033970"
API_URL = f"https://zenodo.org/api/records/{RECORD_ID}"
CHUNK = 64 * 1024 * 1024

In [5]:
def md5sum(path: Path, block: int = 1 << 22) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while b := f.read(block):
            h.update(b)
    return h.hexdigest()

In [6]:
def fetch_chunk(url: str, start: int, end: int, part: Path, retries: int = 30) -> int:
    """Download bytes [start, end] into `part`, resuming a partial part file."""
    want = end - start + 1
    for attempt in range(retries):
        have = part.stat().st_size if part.exists() else 0
        if have == want:
            return want
        if have > want:
            part.unlink()
            have = 0
        try:
            req = urllib.request.Request(url, headers={"Range": f"bytes={start + have}-{end}"})
            with urllib.request.urlopen(req, timeout=60) as resp, open(part, "ab") as f:
                while b := resp.read(1 << 18):
                    f.write(b)
        except Exception as e:  # network hiccup -> back off and resume
            print(f"\n[retry {attempt + 1}] {part.name}: {e}", flush=True)
            time.sleep(min(5 * (attempt + 1), 60))
    raise RuntimeError(f"Failed to download {part.name}")

In [7]:
def parallel_download(url: str, dest: Path, size: int, workers: int) -> None:
    if dest.exists() and dest.stat().st_size == size:
        print(f"[skip] {dest.name} already assembled")
        return
    parts_dir = dest.with_suffix(dest.suffix + ".parts")
    parts_dir.mkdir(exist_ok=True)
    ranges = [(i, s, min(s + CHUNK, size) - 1) for i, s in enumerate(range(0, size, CHUNK))]
    t0, done = time.time(), 0
    with ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(fetch_chunk, url, s, e, parts_dir / f"{i:05d}.part") for i, s, e in ranges]
        for n, fut in enumerate(as_completed(futs), 1):
            done += fut.result()
            rate = done / max(time.time() - t0, 1e-6) / 1e6
            print(f"chunks {n}/{len(ranges)}  ~{rate:.2f} MB/s (this session)", flush=True)

    print(f"Assembling {dest.name} ...", flush=True)
    with open(dest, "wb") as out:
        for i, _, _ in ranges:
            with open(parts_dir / f"{i:05d}.part", "rb") as f:
                while b := f.read(1 << 22):
                    out.write(b)
    assert dest.stat().st_size == size, "assembled size mismatch"
    for p in parts_dir.iterdir():
        p.unlink()
    parts_dir.rmdir()

In [8]:
def main_download_data() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/raw")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--no-extract", action="store_true")
    args = ap.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    with urllib.request.urlopen(API_URL, timeout=60) as r:
        record = json.load(r)
    assert record["doi"] == DOI, f"Unexpected DOI {record['doi']}"
    (out / "zenodo_record.json").write_text(json.dumps(record, indent=2))

    for f in record["files"]:
        dest = out / f["key"]
        algo, expected_md5 = f["checksum"].split(":")
        assert algo == "md5"
        parallel_download(f["links"]["self"], dest, f["size"], args.workers)

        print(f"Verifying MD5 of {dest.name} ...", flush=True)
        got = md5sum(dest)
        if got != expected_md5:
            raise SystemExit(f"MD5 mismatch for {dest.name}: got {got}, expected {expected_md5}")
        print(f"[ok] md5 {got}", flush=True)

        if not args.no_extract and dest.suffix == ".zip":
            print(f"Extracting {dest.name} ...", flush=True)
            with zipfile.ZipFile(dest) as z:
                z.extractall(out)
            print("[ok] extracted", flush=True)

In [9]:
# SKIPPED in this re-run: the raw audio is already present (data/raw/DATASEC -> C:\ATI\DATASEC);
# running the download would re-fetch 6.4 GB and re-extract over the existing dataset.
# sys.argv = ["scripts/download_data.py", "--out", "data/raw", "--workers", "8"]
# main_download_data()
print("download skipped (raw data already present)")

download skipped (raw data already present)


## 2. Index the clips

Source: `scripts/build_index.py`

Scan extracted DataSEC and write data/index.csv with one row per clip.

Columns: sample_id, path, macro_class, subclass, sr, channels, frames, duration, subtype, md5
Exact-duplicate audio (same md5) is reported so splits can keep duplicates in one partition.

In [10]:
import hashlib
import sys
from pathlib import Path
import pandas as pd
import soundfile as sf
from tqdm import tqdm
from src.utils import load_config, load_classes, resolve

In [11]:
def main_build_index() -> None:
    cfg = load_config()
    root = resolve(cfg["dataset"]["raw_dir"]) / "DATASEC"
    classes = load_classes()

    rows = []
    for wav in tqdm(sorted(root.rglob("*.wav")), desc="index"):
        rel = wav.relative_to(root)
        macro = rel.parts[0]
        sub = rel.parts[1] if len(rel.parts) == 3 else macro
        info = sf.info(str(wav))
        rows.append({
            "sample_id": wav.stem,
            "path": rel.as_posix(),
            "macro_class": macro,
            "subclass": sub,
            "sr": info.samplerate,
            "channels": info.channels,
            "frames": info.frames,
            "duration": info.frames / info.samplerate,
            "subtype": info.subtype,
            "md5": hashlib.md5(wav.read_bytes()).hexdigest(),
        })

    df = pd.DataFrame(rows)
    assert df["sample_id"].is_unique, "sample_id (file stem) is not unique"
    assert set(df["macro_class"]) == set(classes), "macro-class folders differ from configs/classes.json"

    out = resolve(cfg["dataset"]["index_csv"])
    df.to_csv(out, index=False)

    print(f"clips={len(df)}  macro={df.macro_class.nunique()}  leaf={df.subclass.nunique()}  "
          f"hours={df.duration.sum() / 3600:.2f}")
    print("sample rates:", df.sr.value_counts().to_dict())
    print("channels:", df.channels.value_counts().to_dict())
    print("subtypes:", df.subtype.value_counts().to_dict())
    dups = df[df.duplicated("md5", keep=False)].sort_values("md5")
    print(f"exact-duplicate clips: {len(dups)} in {dups.md5.nunique()} groups")
    if len(dups):
        dups.to_csv(out.with_name("exact_duplicates.csv"), index=False)
    print(f"wrote {out}")

In [12]:
# same as: python scripts/build_index.py
sys.argv = ["scripts/build_index.py"]
main_build_index()

index:   0%|          | 0/5048 [00:00<?, ?it/s]

index:   0%|          | 21/5048 [00:00<00:24, 206.64it/s]

index:   1%|          | 47/5048 [00:00<00:21, 234.79it/s]

index:   2%|▏         | 109/5048 [00:00<00:12, 405.49it/s]

index:   3%|▎         | 150/5048 [00:00<00:12, 382.02it/s]

...

index:  94%|█████████▍| 4736/5048 [00:11<00:00, 1211.54it/s]

index:  96%|█████████▋| 4864/5048 [00:11<00:00, 832.46it/s] 

index:  98%|█████████▊| 4968/5048 [00:11<00:00, 562.42it/s]

index: 100%|██████████| 5048/5048 [00:12<00:00, 418.26it/s]

clips=5048  macro=22  leaf=40  hours=23.71
sample rates: {44100: 5048}
channels: {1: 5048}
subtypes: {'PCM_16': 5048}
exact-duplicate clips: 32 in 16 groups
wrote C:\ATI\DataSEC-Sound-Classification-main\data\index.csv


## 3. Fixed train / val / test split

Source: `scripts/make_splits.py`

Create the fixed train/val/test split (70/15/15), stratified by leaf class, grouped by duplicate audio.

Stratifying on the leaf class (subclass, or macro-class when it has no subfolders) also keeps
macro-class proportions. Clips sharing a group id (exact md5 duplicates, plus any near-duplicate
groups in data/near_duplicates.csv) always land in the same partition, so no audio leaks into test.

In [13]:
import json
import sys
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from src.utils import load_config, resolve

In [14]:
N_FOLDS = 20  # 1 fold = 5% -> 3 folds test, 3 folds val, 14 folds train

In [15]:
def build_groups(df: pd.DataFrame, near_dup_csv: Path) -> pd.Series:
    group = df["md5"].copy()
    if near_dup_csv.exists():  # columns: sample_id, group
        nd = pd.read_csv(near_dup_csv).set_index("sample_id")["group"]
        hit = df["sample_id"].isin(nd.index)
        group[hit] = "nd_" + df.loc[hit, "sample_id"].map(nd).astype(str)
    return group

In [16]:
def main_make_splits() -> None:
    cfg = load_config()
    seed = cfg["split"]["seed"]
    df = pd.read_csv(resolve(cfg["dataset"]["index_csv"]))
    groups = build_groups(df, resolve("data/near_duplicates.csv"))

    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold = pd.Series(-1, index=df.index)
    for k, (_, idx) in enumerate(sgkf.split(df, df["subclass"], groups)):
        fold.iloc[idx] = k
    df["split"] = fold.map(lambda k: "test" if k < 3 else "val" if k < 6 else "train")

    # sanity: every macro-class in every split, and no group crosses partitions
    per = pd.crosstab(df["macro_class"], df["split"])
    assert (per > 0).all().all(), f"a class is missing from a split:\n{per}"
    assert (df.groupby(groups)["split"].nunique() == 1).all(), "duplicate group crosses splits"

    out_dir = resolve(cfg["dataset"]["splits_dir"])
    out_dir.mkdir(exist_ok=True)
    cols = ["sample_id", "path", "macro_class", "subclass", "duration"]
    for name in ["train", "val", "test"]:
        df.loc[df.split == name, cols].sort_values("sample_id").to_csv(out_dir / f"{name}.csv", index=False)

    meta = {
        "seed": seed,
        "method": f"StratifiedGroupKFold(n_splits={N_FOLDS}) on leaf class; folds 0-2 test, 3-5 val, rest train",
        "groups": "md5 exact duplicates (+ data/near_duplicates.csv if present)",
        "n_groups_multi": int((groups.value_counts() > 1).sum()),
        "sizes": df.split.value_counts().to_dict(),
        "per_macro_class": per.to_dict(orient="index"),
    }
    (out_dir / "split_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    print(per.assign(total=per.sum(axis=1)).to_string())
    print(meta["sizes"])

In [17]:
# The committed splits/ are fixed for the Midterm and the Final: do not regenerate them.
REGENERATE_SPLITS = False
if REGENERATE_SPLITS:
    sys.argv = ["scripts/make_splits.py"]
    main_make_splits()
else:
    meta = pd.read_json("splits/split_meta.json", typ="series")
    print(meta["method"], "|", meta["groups"], "|", meta["sizes"])

StratifiedGroupKFold(n_splits=20) on leaf class; folds 0-2 test, 3-5 val, rest train | md5 exact duplicates (+ data/near_duplicates.csv if present) | {'train': 3532, 'test': 758, 'val': 758}


## 4. Cache 22.05 kHz waveforms

Source: `scripts/cache_waveforms.py`

Resample every clip to the configured rate and cache it as int16 .npy (audio only — no spectrograms).

Speeds up Colab training: reading ~6 GB of 44.1 kHz WAV each epoch is IO-bound.
Amplitude is kept as-is (no peak normalization) so gain perturbations stay meaningful.

In [18]:
import argparse
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import soxr
from tqdm import tqdm
from src.utils import load_config, resolve

In [19]:
def convert(args: tuple[str, str, int]) -> str:
    src, dst, target_sr = args
    if Path(dst).exists():
        return dst
    wav, sr = sf.read(src, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)  # mono (DataSEC is mono already; guard anyway)
    if sr != target_sr:
        wav = soxr.resample(wav, sr, target_sr, quality="HQ")
    np.save(dst, (np.clip(wav, -1.0, 1.0) * 32767).astype(np.int16))
    return dst

In [20]:
def main_cache_waveforms() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=4)
    args = ap.parse_args()

    cfg = load_config()
    sr = cfg["audio"]["sample_rate"]
    raw = resolve(cfg["dataset"]["raw_dir"]) / "DATASEC"
    cache = resolve(cfg["dataset"]["cache_dir"])
    cache.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(resolve(cfg["dataset"]["index_csv"]))
    jobs = [(str(raw / p), str(cache / f"{sid}.npy"), sr) for sid, p in zip(df.sample_id, df.path)]
    with ProcessPoolExecutor(args.workers) as ex:
        list(tqdm(ex.map(convert, jobs, chunksize=8), total=len(jobs), desc=f"cache {sr} Hz"))
    print(f"cached {len(jobs)} clips to {cache}")

In [21]:
# the worker must be importable for multiprocessing on Windows / macOS (spawn)
from scripts.cache_waveforms import convert  # noqa: F811

# same as: python scripts/cache_waveforms.py --workers 4
sys.argv = ["scripts/cache_waveforms.py", "--workers", "4"]
main_cache_waveforms()

cache 22050 Hz:   0%|          | 0/5048 [00:00<?, ?it/s]

cache 22050 Hz:   0%|          | 1/5048 [00:00<33:56,  2.48it/s]

cache 22050 Hz:  95%|█████████▌| 4817/5048 [00:00<00:00, 12590.18it/s]

cache 22050 Hz: 100%|██████████| 5048/5048 [00:00<00:00, 9941.26it/s] 

cached 5048 clips to C:\ATI\DataSEC-Sound-Classification-main\data\cache_22k
